**Preprocessing Pipeline Stunting Kota Semarang**

Transformasi integrated_data_stunting.csv menjadi dua dataset bersih:
  1. stunting_cleaned_monthly.csv       -> trend analysis (Q1)
  2. stunting_agg_puskesmas_yearly.csv  -> correlation & clustering (Q2-Q4)

## LOAD


In [36]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from scipy import stats
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

if not Path("data/integrated").exists() and Path("..").joinpath("data/integrated").exists():
    os.chdir("..")

INPUT_FILE = "data/integrated/integrated_data_stunting.csv"
OUTPUT_DIR = "data/preprocessed"

df = pd.read_csv(INPUT_FILE)
shape_initial = df.shape
print(f"[LOAD] Shape awal: {shape_initial}")
print(f"[LOAD] Kolom: {list(df.columns)}")

log = []

[LOAD] Shape awal: (2148, 63)
[LOAD] Kolom: ['no', 'kecamatan', 'id_kec', 'puskesmas', 'kelurahan', 'jml_posyandu', 'balita_ada', 'bulan', 'tahun', 'total_stunting', 'u0_6', 'u6_11', 'u12_23', 'u24_59', 'kasus_baru', 'lulus', 'kasus_lama', 'total_balita', 'lama_0_6', 'lama_6_11', 'lama_12_23', 'lama_24_59', 'baru_0_6', 'baru_6_11', 'baru_12_23', 'baru_24_59', 'lulus_membaik_0_6', 'lulus_membaik_6_11', 'lulus_membaik_12_23', 'lulus_membaik_24_59', 'meninggal', 'lulus_pindah_0_6', 'lulus_pindah_6_11', 'lulus_pindah_12_23', 'lulus_pindah_24_59', 'lulus_umur_gt59', 'total_lulus', 'kode_puskesmas', 'latitude', 'longitude', 'luas_wilayah_km2', 'jumlah_penduduk', 'jumlah_kk', 'karakteristik_wilayah', 'sanitasi_pilar1', 'sanitasi_pilar2', 'sanitasi_pilar3', 'sanitasi_pilar4', 'sanitasi_pilar5', 'hamil_jarak2th', 'hamil_paritas', 'hamil_umur_lebih35th', 'hamil_umur_kurang20th', 'kunjungan_k4', 'kunjungan_k1', 'kasus_bblr', 'hamil_risti', 'ibu_hamil_total', 'rasio_risti', 'capaian_imunisasi', 'i

## DATA INTEGRATION & SCHEMA HARMONIZATION


### 1.1 Rejoin identitas puskesmas


In [37]:
PUSKESMAS_NAME_MAP = {
    "KARANG MALANG":    "KARANGMALANG",
    "BANGET AYU":       "BANGETAYU",
    "GAYAM SARI":       "GAYAMSARI",
    "KARANG DORO":      "KARANGDORO",
    "KARANG AYU":       "KARANGAYU",
    "KARANG ANYAR":     "KARANGANYAR",
    "KEDUNG MUNDU":     "KEDUNGMUNDU",
    "PUNDAKPAYUNG":     "PUDAKPAYUNG",
    "TELOGOSARI KULON": "TLOGOSARI KULON",
    "TELOGOSARI WETAN": "TLOGOSARI WETAN",
}

identitas_path = "data/raw/identitas-puskesmas-2022-kota-semarang.csv"
df_ident = pd.read_csv(identitas_path, sep=";", on_bad_lines="skip")
df_ident.columns = df_ident.columns.str.strip()
df_ident = df_ident[df_ident["NAMA PUSKESMAS"].notna()].copy()
df_ident["puskesmas_key"] = df_ident["NAMA PUSKESMAS"].str.strip().str.upper()
df_ident["puskesmas_key"] = df_ident["puskesmas_key"].replace(PUSKESMAS_NAME_MAP)

for col_src, col_dst in [("LINTANG", "latitude"), ("BUJUR", "longitude")]:
    df_ident[col_dst] = pd.to_numeric(
        df_ident[col_src].astype(str).str.replace(",", ".").str.strip(),
        errors="coerce"
    )

for col_src, col_dst in [("LUAS WILAYAH", "luas_wilayah_km2"),
                          ("JUMLAH PENDUDUK", "jumlah_penduduk"),
                          ("JUMLAH KK", "jumlah_kk")]:
    if col_src in df_ident.columns:
        df_ident[col_dst] = pd.to_numeric(
            df_ident[col_src].astype(str).str.replace(",", ".").str.strip(),
            errors="coerce"
        )

ident_map = df_ident.set_index("puskesmas_key")[["latitude", "longitude",
    "luas_wilayah_km2", "jumlah_penduduk", "jumlah_kk"]].to_dict("index")

n_fixed_geo = 0
for idx, row in df.iterrows():
    pusk = str(row["puskesmas"]).strip().upper() if pd.notna(row["puskesmas"]) else ""
    if pusk in ident_map:
        info = ident_map[pusk]
        if pd.isna(row.get("latitude")):
            df.at[idx, "latitude"] = info["latitude"]
            df.at[idx, "longitude"] = info["longitude"]
            n_fixed_geo += 1
        for field in ["luas_wilayah_km2", "jumlah_penduduk", "jumlah_kk"]:
            if pd.isna(row.get(field)) and pd.notna(info.get(field)):
                df.at[idx, field] = info[field]

pendukung_cols = {
    'Identitas': ['latitude', 'longitude', 'luas_wilayah_km2', 'jumlah_penduduk'],
    'Sanitasi': ['sanitasi_pilar1'],
    '4T': ['hamil_jarak2th'],
    'K4': ['kunjungan_k4'],
    'K1': ['kunjungan_k1'],
    'BBLR': ['kasus_bblr'],
    'Risti': ['hamil_risti'],
}
mismatch_puskesmas = set()
for source, cols in pendukung_cols.items():
    col = cols[0]
    if col in df.columns:
        missing_mask = df[col].isna()
        missing_pusk = df.loc[missing_mask, 'puskesmas'].dropna().unique()
        mismatch_puskesmas.update(missing_pusk)

all_puskesmas = sorted(df['puskesmas'].dropna().unique())
print(f"Total puskesmas: {len(all_puskesmas)}")
print(f"Puskesmas dengan data pendukung missing: {sorted(mismatch_puskesmas)}")
print(f"Fixed {n_fixed_geo} baris lat/lon dari re-join")
print(f"Lat/lon masih missing: {df['latitude'].isna().sum()}")

log.append(f"1.1 Manual name mapping: {len(PUSKESMAS_NAME_MAP)} puskesmas di-remap. Fixed {n_fixed_geo} baris lat/lon.")

Total puskesmas: 38
Puskesmas dengan data pendukung missing: ['3', 'TAMBAK AJI']
Fixed 588 baris lat/lon dari re-join
Lat/lon masih missing: 24


### 1.2 Rekonsiliasi skema semester


In [38]:
# Jan-Mei: kolom 'kasus_baru'
# Jun-Des: sum dari baru_0_6 + baru_6_11 + baru_12_23 + baru_24_59
baru_detail_cols = ['baru_0_6', 'baru_6_11', 'baru_12_23', 'baru_24_59']
baru_sum = df[baru_detail_cols].sum(axis=1, min_count=1)
df['total_kasus_baru'] = df['kasus_baru'].fillna(baru_sum)

# Jun-Des: sum dari lulus_membaik + lulus_pindah + lulus_umur_gt59
lulus_detail_cols = [
    'lulus_membaik_0_6', 'lulus_membaik_6_11', 'lulus_membaik_12_23', 'lulus_membaik_24_59',
    'lulus_pindah_0_6', 'lulus_pindah_6_11', 'lulus_pindah_12_23', 'lulus_pindah_24_59',
    'lulus_umur_gt59'
]
existing_lulus_cols = [c for c in lulus_detail_cols if c in df.columns]
lulus_sum = df[existing_lulus_cols].sum(axis=1, min_count=1)

# Prefer existing 'total_lulus' for Jun-Des, fallback to computed sum
lulus_semester2 = df['total_lulus'].fillna(lulus_sum) if 'total_lulus' in df.columns else lulus_sum

# Jan-Mei: kolom 'lulus'
df['total_lulus_unified'] = df['lulus'].fillna(lulus_semester2)

print(f"total_kasus_baru: {df['total_kasus_baru'].notna().sum()}/{len(df)} non-null")
print(f"total_lulus_unified: {df['total_lulus_unified'].notna().sum()}/{len(df)} non-null")

log.append("1.2 Rekonsiliasi skema semester: total_kasus_baru, total_lulus_unified")

total_kasus_baru: 1880/2148 non-null
total_lulus_unified: 1909/2148 non-null


## DATA CLEANING

### 2.1 Drop baris spurious

In [39]:
# Header leak: kecamatan == '2', puskesmas == '3'
mask_header = (df['kecamatan'].astype(str).str.strip() == '2') | \
              (df['puskesmas'].astype(str).str.strip() == '3')
n_header = mask_header.sum()

# Baris agregat kota
mask_total = df['kecamatan'].astype(str).str.strip().str.upper() == 'KOTA SEMARANG'
n_total = mask_total.sum()

# Puskesmas NaN
mask_nan_pusk = df['puskesmas'].isna()
n_nan = mask_nan_pusk.sum()

mask_drop = mask_header | mask_total | mask_nan_pusk
n_drop = mask_drop.sum()
df = df[~mask_drop].reset_index(drop=True)

print(f"Header leak: {n_header} baris")
print(f"Baris total KOTA SEMARANG: {n_total} baris")
print(f"Puskesmas NaN: {n_nan} baris")
print(f"TOTAL dropped: {n_drop} baris")
print(f"Shape setelah spurious cleaning: {df.shape}")

log.append(f"2.1 Dropped {n_drop} baris spurious (header leak={n_header}, total kota={n_total}, puskesmas NaN={n_nan})")

Header leak: 12 baris
Baris total KOTA SEMARANG: 12 baris
Puskesmas NaN: 12 baris
TOTAL dropped: 24 baris
Shape setelah spurious cleaning: (2124, 65)


### 2.2 Fix nilai negatif

In [40]:
n_neg = (df['kasus_lama'] < 0).sum()
df.loc[df['kasus_lama'] < 0, 'kasus_lama'] = np.nan
print(f"Fixed {n_neg} nilai negatif pada kasus_lama -> set NaN")

log.append(f"2.2 Fixed {n_neg} nilai negatif pada kasus_lama -> NaN")

Fixed 10 nilai negatif pada kasus_lama -> set NaN


### 2.3 Drop kolom noise

In [41]:
drop_cols = ['no', 'id_kec', 'tahun', 'idl_laki', 'idl_perempuan', 'idl_total']
existing_drop = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=existing_drop, errors='ignore')
print(f"Dropped noise columns: {existing_drop}")
print(f"Shape setelah drop: {df.shape}")

log.append(f"2.3 Dropped kolom noise: {existing_drop}")

Dropped noise columns: ['no', 'id_kec', 'tahun', 'idl_laki', 'idl_perempuan', 'idl_total']
Shape setelah drop: (2124, 59)


## HANDLING MISSING VALUES

### 3.1 Imputasi lat/lon

In [42]:
n_missing_before = df['latitude'].isna().sum()

for col in ['latitude', 'longitude']:
    df[col] = df.groupby('puskesmas')[col].transform(
        lambda x: x.fillna(x.dropna().iloc[0]) if x.dropna().any() else x
    )

n_missing_after = df['latitude'].isna().sum()
print(f"Latitude missing: {n_missing_before} -> {n_missing_after}")

if n_missing_after > 0:
    missing_pusk_geo = df.loc[df['latitude'].isna(), 'puskesmas'].unique()
    log.append(f"3.1 Imputasi lat/lon: {n_missing_before}->{n_missing_after} missing. Puskesmas tanpa koordinat: {list(missing_pusk_geo)}")
else:
    log.append(f"3.1 Imputasi lat/lon: {n_missing_before}->{n_missing_after} missing. Semua terisi.")

Latitude missing: 0 -> 0


### 3.2 Forwardfill capaian_imunisasi

In [43]:
urutan_bulan = ["Januari", "Februari", "Maret", "April", "Mei", "Juni",
                "Juli", "Agustus", "September", "Oktober", "November", "Desember"]
df['bulan'] = pd.Categorical(df['bulan'], categories=urutan_bulan, ordered=True)
df = df.sort_values(['puskesmas', 'kelurahan', 'bulan']).reset_index(drop=True)

n_missing_imun_before = df['capaian_imunisasi'].isna().sum()
df['capaian_imunisasi'] = df.groupby(['puskesmas', 'kelurahan'])['capaian_imunisasi'].ffill()
n_missing_imun_after = df['capaian_imunisasi'].isna().sum()
print(f"capaian_imunisasi missing: {n_missing_imun_before} -> {n_missing_imun_after}")

log.append(f"3.2 Forward-fill capaian_imunisasi: {n_missing_imun_before}->{n_missing_imun_after} missing")

capaian_imunisasi missing: 199 -> 24


### 3.3 Dokumentasi kolom semesterspesifik

In [44]:
semester1_only = ['u0_6', 'u6_11', 'u12_23', 'u24_59', 'kasus_baru', 'lulus', 'kasus_lama']
semester2_only = ['lama_0_6', 'lama_6_11', 'lama_12_23', 'lama_24_59',
                  'baru_0_6', 'baru_6_11', 'baru_12_23', 'baru_24_59',
                  'lulus_membaik_0_6', 'lulus_membaik_6_11', 'lulus_membaik_12_23', 'lulus_membaik_24_59',
                  'lulus_pindah_0_6', 'lulus_pindah_6_11', 'lulus_pindah_12_23', 'lulus_pindah_24_59',
                  'lulus_umur_gt59', 'meninggal', 'total_lulus', 'total_balita']

existing_s1 = [c for c in semester1_only if c in df.columns]
existing_s2 = [c for c in semester2_only if c in df.columns]
print(f"Semester 1 only (NaN di Jun-Des): {len(existing_s1)} kolom")
print(f"Semester 2 only (NaN di Jan-Mei): {len(existing_s2)} kolom")

log.append(f"3.3 Semester-specific columns: {len(existing_s1)}+{len(existing_s2)} kolom, NaN dipertahankan")

Semester 1 only (NaN di Jun-Des): 7 kolom
Semester 2 only (NaN di Jan-Mei): 20 kolom


## FEATURE ENGINEERING

### 4.1 prevalensi_stunting

In [45]:
df['prevalensi_stunting'] = (df['total_stunting'] / df['balita_ada'].replace(0, np.nan)).clip(upper=1.0)
print(f"prevalensi_stunting: range={df['prevalensi_stunting'].min():.4f} -- {df['prevalensi_stunting'].max():.4f}")

prevalensi_stunting: range=0.0000 -- 1.0000


### 4.2 Fitur Lainnya

In [46]:
# rasio_bblr (per 1000 penduduk)
df['rasio_bblr'] = df['kasus_bblr'] / df['jumlah_penduduk'].replace(0, np.nan) * 1000
print(f"rasio_bblr: mean={df['rasio_bblr'].mean():.3f}")

# cakupan_k4
df['cakupan_k4'] = df['kunjungan_k4'] / df['ibu_hamil_total'].replace(0, np.nan)
print(f"cakupan_k4: mean={df['cakupan_k4'].mean():.3f}")

# densitas_posyandu (per 1000 penduduk)
df['densitas_posyandu'] = df['jml_posyandu'] / df['jumlah_penduduk'].replace(0, np.nan) * 1000
print(f"densitas_posyandu: mean={df['densitas_posyandu'].mean():.3f}")

# sanitasi_total (rata-rata 5 pilar)
pilar_cols = ['sanitasi_pilar1', 'sanitasi_pilar2', 'sanitasi_pilar3', 'sanitasi_pilar4', 'sanitasi_pilar5']
existing_pilar = [c for c in pilar_cols if c in df.columns]
df['sanitasi_total'] = df[existing_pilar].mean(axis=1)
print(f"sanitasi_total: mean={df['sanitasi_total'].mean():.1f}")

# risiko_ibu_komposit (equal weight: rata-rata rasio 4T dan rasio risti)
risiko_4t_cols = ['hamil_jarak2th', 'hamil_paritas', 'hamil_umur_lebih35th', 'hamil_umur_kurang20th']
existing_risiko = [c for c in risiko_4t_cols if c in df.columns]
df['risiko_4t_total'] = df[existing_risiko].sum(axis=1, min_count=1)
rasio_4t = df['risiko_4t_total'] / df['ibu_hamil_total'].replace(0, np.nan)
df['risiko_ibu_komposit'] = (rasio_4t + df['rasio_risti']) / 2  # equal weight
print(f"risiko_ibu_komposit: mean={df['risiko_ibu_komposit'].mean():.4f}")

# tren_stunting (slope linear, minimal 6 bulan data valid)
def calc_slope(group):
    """Hitung slope linear dari total_stunting over 12 bulan. Min 6 bulan."""
    valid = group.dropna(subset=['total_stunting'])
    if len(valid) <= 5: 
        return np.nan
    bulan_num = valid['bulan'].cat.codes.values.astype(float)
    y = valid['total_stunting'].values.astype(float)
    try:
        slope, _, _, _, _ = stats.linregress(bulan_num, y)
        return slope
    except Exception:
        return np.nan

tren = df.groupby('kelurahan', observed=True).apply(calc_slope, include_groups=False).reset_index()
tren.columns = ['kelurahan', 'tren_stunting']
df = df.merge(tren, on='kelurahan', how='left')

n_kel_tren = tren['tren_stunting'].notna().sum()
n_kel_total = tren.shape[0]
print(f"tren_stunting: {n_kel_tren}/{n_kel_total} kelurahan memiliki tren valid")

log.append(
    f"4.1-4.2 Feature engineering: prevalensi_stunting, rasio_bblr, cakupan_k4, "
    f"densitas_posyandu, sanitasi_total, risiko_ibu_komposit, tren_stunting "
    f"({n_kel_tren}/{n_kel_total} kelurahan valid untuk tren)"
)

rasio_bblr: mean=277.422
cakupan_k4: mean=0.945
densitas_posyandu: mean=205.314
sanitasi_total: mean=22631.5
risiko_ibu_komposit: mean=0.2643
tren_stunting: 175/175 kelurahan memiliki tren valid


## OUTPUT

In [47]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

### 5.1 Output bulanan

In [48]:
monthly_cols = [
    'kecamatan', 'puskesmas', 'kelurahan', 'bulan',
    'jml_posyandu', 'balita_ada',
    'total_stunting', 'total_kasus_baru', 'total_lulus_unified',
    'prevalensi_stunting', 'tren_stunting',
    'capaian_imunisasi',
    'latitude', 'longitude'
]
existing_monthly = [c for c in monthly_cols if c in df.columns]

df_monthly = df[existing_monthly].copy()
df_monthly = df_monthly.sort_values(['kelurahan', 'bulan']).reset_index(drop=True)

monthly_path = os.path.join(OUTPUT_DIR, "stunting_cleaned_monthly.csv")
df_monthly.to_csv(monthly_path, index=False)
print(f"Output bulanan saved: {monthly_path} (shape: {df_monthly.shape})")

Output bulanan saved: data/preprocessed\stunting_cleaned_monthly.csv (shape: (2124, 14))


### 5.2 Output tahunan per puskesmas

In [49]:
sum_nan = lambda s: s.sum(min_count=1)
agg_dict = {
    # Stunting 
    'total_stunting': sum_nan,
    'balita_ada': sum_nan,
    'total_kasus_baru': sum_nan,
    'total_lulus_unified': sum_nan,
    'tren_stunting': 'median',   

    # Geospasial
    'latitude': 'first',
    'longitude': 'first',
    'kecamatan': 'first',

    # Demografi
    'jumlah_penduduk': 'first',
    'jumlah_kk': 'first',
    'luas_wilayah_km2': 'first',
    'karakteristik_wilayah': 'first',

    # Sanitasi
    'sanitasi_total': 'first',
    'sanitasi_pilar1': 'first',
    'sanitasi_pilar2': 'first',
    'sanitasi_pilar3': 'first',
    'sanitasi_pilar4': 'first',
    'sanitasi_pilar5': 'first',

    # Kesehatan ibu
    'hamil_risti': 'first',
    'ibu_hamil_total': 'first',
    'rasio_risti': 'first',
    'kunjungan_k1': 'first',
    'kunjungan_k4': 'first',
    'cakupan_k4': 'first',
    'risiko_ibu_komposit': 'first',

    # Bayi
    'kasus_bblr': 'first',
    'rasio_bblr': 'first',

    # Posyandu
    'jml_posyandu': sum_nan,  

    # Imunisasi
    'capaian_imunisasi': 'mean',
}

valid_agg = {k: v for k, v in agg_dict.items() if k in df.columns}
df_yearly = df.groupby('puskesmas', observed=True).agg(valid_agg).reset_index()
df_yearly['prevalensi_stunting'] = (
    df_yearly['total_stunting'] / df_yearly['balita_ada']
)
df_yearly['densitas_posyandu'] = (
    df_yearly['jml_posyandu'] / df_yearly['jumlah_penduduk'].replace(0, np.nan) * 1000
)
df_yearly = df_yearly.sort_values('prevalensi_stunting', ascending=False).reset_index(drop=True)
yearly_path = os.path.join(OUTPUT_DIR, "stunting_agg_puskesmas_yearly.csv")
df_yearly.to_csv(yearly_path, index=False)

print(f"Output tahunan saved: {yearly_path} (shape: {df_yearly.shape})")
print("\nTop 5 puskesmas (prevalensi stunting tertinggi):")
print(df_yearly[['puskesmas', 'prevalensi_stunting', 'total_stunting', 'balita_ada']].head().to_string(index=False))
print("\nBottom 5 puskesmas (prevalensi stunting terendah):")
print(df_yearly[['puskesmas', 'prevalensi_stunting', 'total_stunting', 'balita_ada']].tail().to_string(index=False))

Output tahunan saved: data/preprocessed\stunting_agg_puskesmas_yearly.csv (shape: (37, 32))

Top 5 puskesmas (prevalensi stunting tertinggi):
    puskesmas  prevalensi_stunting  total_stunting  balita_ada
 KARANGMALANG             0.909091             480       528.0
       PONCOL             0.061964             925     14928.0
   KARANGDORO             0.050779             900     17724.0
LAMPER TENGAH             0.046507             663     14256.0
  BANDARHARJO             0.045717            2545     55668.0

Bottom 5 puskesmas (prevalensi stunting terendah):
puskesmas  prevalensi_stunting  total_stunting  balita_ada
 NGALIYAN             0.003902             212     54336.0
BANGETAYU             0.002829             224     79188.0
CANDILAMA                  NaN             407         NaN
    KAGOK                  NaN              52         NaN
 MANGKANG                  NaN             231         NaN


In [50]:
log_path = os.path.join(OUTPUT_DIR, "preprocessing_log.txt")
with open(log_path, "w") as f:
    f.write("\n".join(log))
print(f"Log saved: {log_path}")

Log saved: data/preprocessed\preprocessing_log.txt
